In [2]:
from pathlib import Path

DATA_PATH = Path("../data/processed")

list(DATA_PATH.glob("*"))

[WindowsPath('../data/processed/sparkov_anomaly_scores.parquet'),
 WindowsPath('../data/processed/sparkov_features.parquet'),
 WindowsPath('../data/processed/sparkov_risk_engine.parquet')]

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
anomaly_df = pd.read_parquet(
    DATA_PATH / "sparkov_anomaly_scores.parquet"
)

features_df = pd.read_parquet(
    DATA_PATH / "sparkov_features.parquet"
)

print("Anomaly dataset shape:", anomaly_df.shape)
print("Features dataset shape:", features_df.shape)

Anomaly dataset shape: (1296675, 27)
Features dataset shape: (1296675, 25)


In [4]:
print("Anomaly columns:")
print(anomaly_df.columns.tolist())

print("\nFeature columns:")
print(features_df.columns.tolist())

Anomaly columns:
['transaction_time', 'cc_num', 'trans_num', 'amt', 'log_amount', 'hour', 'day_of_week', 'month', 'day_of_month', 'is_weekend', 'customer_prev_count', 'customer_prev_avg_amount', 'customer_prev_median_amount', 'customer_prev_std_amount', 'amount_deviation_ratio', 'seconds_since_prev_transaction', 'transactions_prev_5min', 'transactions_prev_1h', 'transactions_prev_24h', 'merchant_seen_before', 'is_new_merchant', 'category_seen_before', 'is_new_category', 'customer_merchant_distance_km', 'is_fraud', 'anomaly_score', 'anomaly_flag']

Feature columns:
['transaction_time', 'cc_num', 'trans_num', 'amt', 'log_amount', 'hour', 'day_of_week', 'month', 'day_of_month', 'is_weekend', 'customer_prev_count', 'customer_prev_avg_amount', 'customer_prev_median_amount', 'customer_prev_std_amount', 'amount_deviation_ratio', 'seconds_since_prev_transaction', 'transactions_prev_5min', 'transactions_prev_1h', 'transactions_prev_24h', 'merchant_seen_before', 'is_new_merchant', 'category_seen

In [5]:
display(anomaly_df.head())
display(features_df.head())

,transaction_time,cc_num,trans_num,amt,log_amount,hour,day_of_week,month,day_of_month,is_weekend,...,transactions_prev_1h,transactions_prev_24h,merchant_seen_before,is_new_merchant,category_seen_before,is_new_category,customer_merchant_distance_km,is_fraud,anomaly_score,anomaly_flag
0,2019-01-01 12:47:15,60416207185,98e3dcf98101146a577f85a34e58feec,7.27,2.112635,12,1,1,1,0,...,NaN,NaN,0,1,0,1,127.606239,0,0.027990,0
1,2019-01-02 08:44:57,60416207185,498120fc45d277f7c88e3dba79c33865,52.94,3.987872,8,2,1,2,0,...,NaN,1.0,0,1,0,1,110.308921,0,0.073429,0
2,2019-01-02 08:47:36,60416207185,95f514bb993151347c7acdf8505c3d62,82.08,4.419804,8,2,1,2,0,...,1.0,2.0,0,1,1,0,21.787261,0,-0.003784,0
3,2019-01-02 12:38:14,60416207185,4f0c1a14e0aa7eb56a490780ef9268c5,34.79,3.577669,12,2,1,2,0,...,NaN,3.0,0,1,0,1,87.204215,0,0.011825,0
4,2019-01-02 13:10:46,60416207185,3b2ebd3af508afba959640893e1e82bc,27.18,3.338613,13,2,1,2,0,...,1.0,3.0,0,1,0,1,74.212965,0,0.016785,0


,transaction_time,cc_num,trans_num,amt,log_amount,hour,day_of_week,month,day_of_month,is_weekend,...,seconds_since_prev_transaction,transactions_prev_5min,transactions_prev_1h,transactions_prev_24h,merchant_seen_before,is_new_merchant,category_seen_before,is_new_category,customer_merchant_distance_km,is_fraud
0,2019-01-01 12:47:15,60416207185,98e3dcf98101146a577f85a34e58feec,7.27,2.112635,12,1,1,1,0,...,NaN,NaN,NaN,NaN,0,1,0,1,127.606239,0
1,2019-01-02 08:44:57,60416207185,498120fc45d277f7c88e3dba79c33865,52.94,3.987872,8,2,1,2,0,...,71862.0,NaN,NaN,1.0,0,1,0,1,110.308921,0
2,2019-01-02 08:47:36,60416207185,95f514bb993151347c7acdf8505c3d62,82.08,4.419804,8,2,1,2,0,...,159.0,1.0,1.0,2.0,0,1,1,0,21.787261,0
3,2019-01-02 12:38:14,60416207185,4f0c1a14e0aa7eb56a490780ef9268c5,34.79,3.577669,12,2,1,2,0,...,13838.0,NaN,NaN,3.0,0,1,0,1,87.204215,0
4,2019-01-02 13:10:46,60416207185,3b2ebd3af508afba959640893e1e82bc,27.18,3.338613,13,2,1,2,0,...,1952.0,NaN,1.0,3.0,0,1,0,1,74.212965,0


In [6]:
# Identify the key risk-related columns

risk_columns = [
    col for col in anomaly_df.columns
    if any(keyword in col.lower() for keyword in [
        "anomaly",
        "behaviour",
        "risk",
        "fraud",
        "signal"
    ])
]

print("Risk-related columns:")
for col in risk_columns:
    print("-", col)

Risk-related columns:
- is_fraud
- anomaly_score
- anomaly_flag


In [7]:
# Check whether the two datasets have the same transaction ordering

print(
    "Transaction IDs identical:",
    anomaly_df["trans_num"].equals(features_df["trans_num"])
)

print(
    "Row count identical:",
    len(anomaly_df) == len(features_df)
)

Transaction IDs identical: True
Row count identical: True


In [8]:
anomaly_df[
    [col for col in anomaly_df.columns
     if "anomaly" in col.lower()]
].head()

,anomaly_score,anomaly_flag
0,0.027990,0
1,0.073429,0
2,-0.003784,0
3,0.011825,0
4,0.016785,0


## Phase 9 — Risk Engine Construction

This phase integrates the behavioural risk indicators developed in Phase 8 with the unsupervised anomaly-detection signal developed in Phase 7.

The objective is to construct a unified transaction-level risk score that combines complementary behavioural and anomaly signals while retaining the original fraud label for validation.

In [9]:
# Combine Phase 8 behavioural features with Phase 7 anomaly outputs
# using the transaction identifier rather than row position.

risk_df = features_df.merge(
    anomaly_df[
        [
            "trans_num",
            "anomaly_score",
            "anomaly_flag"
        ]
    ],
    on="trans_num",
    how="inner",
    validate="one_to_one"
)

print("Risk engine dataset shape:", risk_df.shape)

print("\nRisk engine columns:")
print(risk_df.columns.tolist())

Risk engine dataset shape: (1296675, 27)

Risk engine columns:
['transaction_time', 'cc_num', 'trans_num', 'amt', 'log_amount', 'hour', 'day_of_week', 'month', 'day_of_month', 'is_weekend', 'customer_prev_count', 'customer_prev_avg_amount', 'customer_prev_median_amount', 'customer_prev_std_amount', 'amount_deviation_ratio', 'seconds_since_prev_transaction', 'transactions_prev_5min', 'transactions_prev_1h', 'transactions_prev_24h', 'merchant_seen_before', 'is_new_merchant', 'category_seen_before', 'is_new_category', 'customer_merchant_distance_km', 'is_fraud', 'anomaly_score', 'anomaly_flag']


In [10]:
behaviour_columns = [
    col for col in risk_df.columns
    if "behaviour" in col.lower()
]

print("Behavioural risk columns:")
for col in behaviour_columns:
    print("-", col)

Behavioural risk columns:


In [11]:
# Check all columns available in the features dataset

for i, col in enumerate(features_df.columns, start=1):
    print(i, col)

1 transaction_time
2 cc_num
3 trans_num
4 amt
5 log_amount
6 hour
7 day_of_week
8 month
9 day_of_month
10 is_weekend
11 customer_prev_count
12 customer_prev_avg_amount
13 customer_prev_median_amount
14 customer_prev_std_amount
15 amount_deviation_ratio
16 seconds_since_prev_transaction
17 transactions_prev_5min
18 transactions_prev_1h
19 transactions_prev_24h
20 merchant_seen_before
21 is_new_merchant
22 category_seen_before
23 is_new_category
24 customer_merchant_distance_km
25 is_fraud


In [12]:
# Check whether any behavioural signal columns are present

signal_columns = [
    col for col in features_df.columns
    if any(x in col.lower() for x in ["signal", "velocity", "novelty", "behaviour"])
]

print("Signal-related columns:")
for col in signal_columns:
    print("-", col)

Signal-related columns:


## 3. Define Risk Engine Inputs

The risk engine combines the Phase 7 anomaly-detection outputs with the behavioural
features developed during Phase 8.

The available inputs are:

- Anomaly score
- Anomaly flag
- Amount deviation
- Transaction velocity
- Merchant/category novelty
- Geographic distance
- Transaction timing and customer history features

The transaction identifier and row ordering have been validated before combining
the datasets.

In [13]:
# Define the core risk-engine inputs

ANOMALY_COLUMNS = [
    "anomaly_score",
    "anomaly_flag"
]

BEHAVIOUR_COLUMNS = [
    "amount_deviation_ratio",
    "transactions_prev_5min",
    "transactions_prev_1h",
    "transactions_prev_24h",
    "is_new_merchant",
    "is_new_category",
    "customer_merchant_distance_km"
]

available_anomaly_columns = [
    col for col in ANOMALY_COLUMNS
    if col in risk_df.columns
]

available_behaviour_columns = [
    col for col in BEHAVIOUR_COLUMNS
    if col in risk_df.columns
]

print("Available anomaly inputs:")
for col in available_anomaly_columns:
    print(" -", col)

print("\nAvailable behavioural inputs:")
for col in available_behaviour_columns:
    print(" -", col)

Available anomaly inputs:
 - anomaly_score
 - anomaly_flag

Available behavioural inputs:
 - amount_deviation_ratio
 - transactions_prev_5min
 - transactions_prev_1h
 - transactions_prev_24h
 - is_new_merchant
 - is_new_category
 - customer_merchant_distance_km


In [14]:
# Verify that all expected risk-engine inputs are available

missing_anomaly = [
    col for col in ANOMALY_COLUMNS
    if col not in risk_df.columns
]

missing_behaviour = [
    col for col in BEHAVIOUR_COLUMNS
    if col not in risk_df.columns
]

print("Missing anomaly inputs:", missing_anomaly)
print("Missing behavioural inputs:", missing_behaviour)

Missing anomaly inputs: []
Missing behavioural inputs: []


## 4. Behavioural Risk Component

The behavioural component aggregates three validated behavioural domains developed
in Phase 8:

1. Amount deviation
2. Transaction velocity
3. Merchant/category novelty

Each domain is converted into a normalized risk contribution before being combined
into a single behavioural risk score.

The component is designed to capture transaction-level behavioural deviation while
remaining interpretable and consistent with the Phase 8 behavioural analysis.

In [15]:
# ---------------------------------------------------------
# Phase 9 — Corrected Behavioural Risk Component
# ---------------------------------------------------------

# Missing historical information is treated as no observed
# behavioural deviation for the corresponding component.

amount_raw = (
    risk_df["amount_deviation_ratio"]
    .fillna(0)
    .clip(lower=0)
)

velocity_raw = (
    risk_df[
        [
            "transactions_prev_5min",
            "transactions_prev_1h",
            "transactions_prev_24h"
        ]
    ]
    .fillna(0)
    .max(axis=1)
)

novelty_raw = (
    risk_df["is_new_merchant"]
    .fillna(0)
    .astype(float)
    +
    risk_df["is_new_category"]
    .fillna(0)
    .astype(float)
)


# ---------------------------------------------------------
# Percentile normalization
# ---------------------------------------------------------

def percentile_score(series, percentile=0.99):
    upper = series.quantile(percentile)

    if upper <= 0:
        return pd.Series(0.0, index=series.index)

    return (
        series
        .clip(lower=0, upper=upper)
        / upper
        * 100
    )


risk_df["amount_risk"] = percentile_score(amount_raw)

risk_df["velocity_risk"] = percentile_score(velocity_raw)

risk_df["novelty_risk"] = percentile_score(novelty_raw)


# ---------------------------------------------------------
# Combined behavioural risk score
# ---------------------------------------------------------

risk_df["behavioural_risk_score"] = (
    risk_df["amount_risk"]
    + risk_df["velocity_risk"]
    + risk_df["novelty_risk"]
) / 3

risk_df["behavioural_risk_score"] = (
    risk_df["behavioural_risk_score"]
    .clip(0, 100)
)


print("Behavioural risk component created successfully.")

print(
    risk_df[
        [
            "amount_risk",
            "velocity_risk",
            "novelty_risk",
            "behavioural_risk_score"
        ]
    ].describe()
)

print("\nMissing values:")
print(
    risk_df[
        [
            "amount_risk",
            "velocity_risk",
            "novelty_risk",
            "behavioural_risk_score"
        ]
    ].isna().sum()
)

Behavioural risk component created successfully.
        amount_risk  velocity_risk  novelty_risk  behavioural_risk_score
count  1.296675e+06   1.296675e+06  1.296675e+06            1.296675e+06
mean   1.248285e+01   2.755690e+01  1.897777e+01            1.967251e+01
std    1.527373e+01   2.123075e+01  2.528212e+01            1.109297e+01
min    0.000000e+00   0.000000e+00  0.000000e+00            3.873390e-02
25%    2.099834e+00   1.428571e+01  0.000000e+00            1.066581e+01
50%    8.935862e+00   2.142857e+01  0.000000e+00            1.921954e+01
75%    1.612764e+01   3.571429e+01  5.000000e+01            2.665519e+01
max    1.000000e+02   1.000000e+02  1.000000e+02            8.968753e+01

Missing values:
amount_risk               0
velocity_risk             0
novelty_risk              0
behavioural_risk_score    0
dtype: int64


In [16]:
risk_df[
    [
        "trans_num",
        "amount_risk",
        "velocity_risk",
        "novelty_risk",
        "behavioural_risk_score",
        "is_fraud"
    ]
].head(20)

,trans_num,amount_risk,velocity_risk,novelty_risk,behavioural_risk_score,is_fraud
0,98e3dcf98101146a577f85a34e58feec,0.000000,0.000000,100.0,33.333333,0
1,498120fc45d277f7c88e3dba79c33865,97.611425,7.142857,100.0,68.251427,0
2,95f514bb993151347c7acdf8505c3d62,36.546841,14.285714,50.0,33.610852,0
3,4f0c1a14e0aa7eb56a490780ef9268c5,9.832239,21.428571,100.0,43.753604,0
4,3b2ebd3af508afba959640893e1e82bc,8.229834,21.428571,100.0,43.219468,0
5,c2c69214de58aaf2bad1542b91751f5a,2.254211,0.000000,100.0,34.084737,0
6,da5ef053fa971418ab30fc72509c66f8,3.211294,7.142857,100.0,36.784717,0
7,79a66e4565825e428e2e098ffd6b9969,50.048369,7.142857,50.0,35.730409,0
8,6dbbe7c58049b9b5cf6c0a29793468b0,8.517225,7.142857,100.0,38.553361,0
9,38dcb937f9e60e26fca7e7f3b3437e07,34.923077,14.285714,100.0,49.736264,0


In [17]:
# ---------------------------------------------------------
# Anomaly Risk Component
# ---------------------------------------------------------
# Convert anomaly_score into a 0–100 risk scale.
# Higher anomaly_score = higher anomaly risk.

anomaly_min = risk_df["anomaly_score"].min()
anomaly_max = risk_df["anomaly_score"].max()

if anomaly_max == anomaly_min:
    risk_df["anomaly_risk"] = 0.0
else:
    risk_df["anomaly_risk"] = (
        (risk_df["anomaly_score"] - anomaly_min)
        / (anomaly_max - anomaly_min)
        * 100
    )

risk_df["anomaly_risk"] = (
    risk_df["anomaly_risk"]
    .clip(0, 100)
)

print("Anomaly risk created successfully.")

print(
    risk_df[
        [
            "anomaly_score",
            "anomaly_flag",
            "anomaly_risk"
        ]
    ].describe()
)

print("\nCorrelation with fraud:")
print(
    risk_df[
        ["anomaly_risk", "is_fraud"]
    ].corr(numeric_only=True)
)

Anomaly risk created successfully.
       anomaly_score  anomaly_flag  anomaly_risk
count   1.296675e+06  1.296675e+06  1.296675e+06
mean   -4.350442e-02  5.000482e-03  2.403769e+01
std     3.477621e-02  7.053709e-02  1.119302e+01
min    -1.181885e-01  0.000000e+00  0.000000e+00
25%    -6.771029e-02  0.000000e+00  1.624682e+01
50%    -4.960918e-02  0.000000e+00  2.207282e+01
75%    -2.663588e-02  0.000000e+00  2.946697e+01
max     1.925071e-01  1.000000e+00  1.000000e+02

Correlation with fraud:
              anomaly_risk  is_fraud
anomaly_risk      1.000000  0.184428
is_fraud          0.184428  1.000000


In [18]:
risk_df[
    [
        "trans_num",
        "anomaly_score",
        "anomaly_flag",
        "anomaly_risk",
        "amount_risk",
        "velocity_risk",
        "novelty_risk",
        "behavioural_risk_score",
        "is_fraud"
    ]
].head(20)

,trans_num,anomaly_score,anomaly_flag,anomaly_risk,amount_risk,velocity_risk,novelty_risk,behavioural_risk_score,is_fraud
0,98e3dcf98101146a577f85a34e58feec,0.027990,0,47.048750,0.000000,0.000000,100.0,33.333333,0
1,498120fc45d277f7c88e3dba79c33865,0.073429,0,61.673698,97.611425,7.142857,100.0,68.251427,0
2,95f514bb993151347c7acdf8505c3d62,-0.003784,0,36.822051,36.546841,14.285714,50.0,33.610852,0
3,4f0c1a14e0aa7eb56a490780ef9268c5,0.011825,0,41.846073,9.832239,21.428571,100.0,43.753604,0
4,3b2ebd3af508afba959640893e1e82bc,0.016785,0,43.442481,8.229834,21.428571,100.0,43.219468,0
5,c2c69214de58aaf2bad1542b91751f5a,0.045291,0,52.617134,2.254211,0.000000,100.0,34.084737,0
6,da5ef053fa971418ab30fc72509c66f8,0.041111,0,51.271996,3.211294,7.142857,100.0,36.784717,0
7,79a66e4565825e428e2e098ffd6b9969,-0.009218,0,35.073009,50.048369,7.142857,50.0,35.730409,0
8,6dbbe7c58049b9b5cf6c0a29793468b0,0.024039,0,45.776998,8.517225,7.142857,100.0,38.553361,0
9,38dcb937f9e60e26fca7e7f3b3437e07,0.059837,0,57.299020,34.923077,14.285714,100.0,49.736264,0


In [19]:
# Combine behavioural and anomaly risk into a final risk score.

risk_df["final_risk_score"] = (
    0.60 * risk_df["behavioural_risk_score"]
    + 0.40 * risk_df["anomaly_risk"]
)

# Ensure the final score remains within 0–100
risk_df["final_risk_score"] = risk_df["final_risk_score"].clip(0, 100)

print("Final risk score created successfully.")

print(
    risk_df["final_risk_score"].describe()
)

Final risk score created successfully.
count    1.296675e+06
mean     2.141858e+01
std      9.378242e+00
min      3.071002e+00
25%      1.439501e+01
50%      2.050995e+01
75%      2.618555e+01
max      8.690866e+01
Name: final_risk_score, dtype: float64


In [20]:
# ---------------------------------------------------------
# Risk Band Assignment
# ---------------------------------------------------------

def assign_risk_band(score):
    if score < 30:
        return "Low"
    elif score < 60:
        return "Medium"
    elif score < 80:
        return "High"
    else:
        return "Critical"


risk_df["risk_band"] = (
    risk_df["final_risk_score"]
    .apply(assign_risk_band)
)

# Preserve the intended severity order
risk_band_order = [
    "Low",
    "Medium",
    "High",
    "Critical"
]

risk_df["risk_band"] = pd.Categorical(
    risk_df["risk_band"],
    categories=risk_band_order,
    ordered=True
)

print("Risk bands created successfully.")

print(
    risk_df["risk_band"]
    .value_counts()
    .sort_index()
)

Risk bands created successfully.
risk_band
Low         1106476
Medium       186512
High           3671
Critical         16
Name: count, dtype: int64


In [21]:
# Flag transactions requiring investigation.

risk_df["investigation_flag"] = (
    (risk_df["final_risk_score"] >= 60)
    | (risk_df["anomaly_flag"] == 1)
)

print("Investigation flags created successfully.")

print(
    risk_df["investigation_flag"].value_counts()
)

Investigation flags created successfully.
investigation_flag
False    1288854
True        7821
Name: count, dtype: int64


In [22]:
risk_df[
    [
        "trans_num",
        "anomaly_score",
        "anomaly_flag",
        "anomaly_risk",
        "amount_risk",
        "velocity_risk",
        "novelty_risk",
        "behavioural_risk_score",
        "final_risk_score",
        "risk_band",
        "investigation_flag",
        "is_fraud"
    ]
].head(20)

,trans_num,anomaly_score,anomaly_flag,anomaly_risk,amount_risk,velocity_risk,novelty_risk,behavioural_risk_score,final_risk_score,risk_band,investigation_flag,is_fraud
0,98e3dcf98101146a577f85a34e58feec,0.027990,0,47.048750,0.000000,0.000000,100.0,33.333333,38.819500,Medium,False,0
1,498120fc45d277f7c88e3dba79c33865,0.073429,0,61.673698,97.611425,7.142857,100.0,68.251427,65.620336,High,True,0
2,95f514bb993151347c7acdf8505c3d62,-0.003784,0,36.822051,36.546841,14.285714,50.0,33.610852,34.895332,Medium,False,0
3,4f0c1a14e0aa7eb56a490780ef9268c5,0.011825,0,41.846073,9.832239,21.428571,100.0,43.753604,42.990591,Medium,False,0
4,3b2ebd3af508afba959640893e1e82bc,0.016785,0,43.442481,8.229834,21.428571,100.0,43.219468,43.308674,Medium,False,0
5,c2c69214de58aaf2bad1542b91751f5a,0.045291,0,52.617134,2.254211,0.000000,100.0,34.084737,41.497696,Medium,False,0
6,da5ef053fa971418ab30fc72509c66f8,0.041111,0,51.271996,3.211294,7.142857,100.0,36.784717,42.579629,Medium,False,0
7,79a66e4565825e428e2e098ffd6b9969,-0.009218,0,35.073009,50.048369,7.142857,50.0,35.730409,35.467449,Medium,False,0
8,6dbbe7c58049b9b5cf6c0a29793468b0,0.024039,0,45.776998,8.517225,7.142857,100.0,38.553361,41.442815,Medium,False,0
9,38dcb937f9e60e26fca7e7f3b3437e07,0.059837,0,57.299020,34.923077,14.285714,100.0,49.736264,52.761366,Medium,False,0


In [23]:
# Compare fraud rate across risk bands.

risk_band_summary = (
    risk_df
    .groupby("risk_band", observed=True)
    .agg(
        transactions=("trans_num", "count"),
        fraud_cases=("is_fraud", "sum"),
        avg_risk_score=("final_risk_score", "mean")
    )
    .reset_index()
)

risk_band_summary["fraud_rate"] = (
    risk_band_summary["fraud_cases"]
    / risk_band_summary["transactions"]
)

risk_band_summary

,risk_band,transactions,fraud_cases,avg_risk_score,fraud_rate
0,Low,1106476,1525,18.556947,0.001378
1,Medium,186512,4833,37.525769,0.025913
2,High,3671,1139,65.320803,0.310270
3,Critical,16,9,82.509340,0.562500


In [24]:
# Examine the relationship between final risk score and fraud.

print("Final risk score distribution:")
print(risk_df["final_risk_score"].describe())

print("\nFraud rate by score ranges:")

risk_score_bins = pd.cut(
    risk_df["final_risk_score"],
    bins=[0, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    include_lowest=True
)

score_summary = (
    risk_df
    .groupby(risk_score_bins, observed=True)
    .agg(
        transactions=("trans_num", "count"),
        fraud_cases=("is_fraud", "sum"),
        avg_risk_score=("final_risk_score", "mean")
    )
    .reset_index()
)

score_summary["fraud_rate"] = (
    score_summary["fraud_cases"]
    / score_summary["transactions"]
)

score_summary

Final risk score distribution:
count    1.296675e+06
mean     2.141858e+01
std      9.378242e+00
min      3.071002e+00
25%      1.439501e+01
50%      2.050995e+01
75%      2.618555e+01
max      8.690866e+01
Name: final_risk_score, dtype: float64

Fraud rate by score ranges:


,final_risk_score,transactions,fraud_cases,avg_risk_score,fraud_rate
0,"(-0.001, 20.0]",617507,642,13.997599,0.001040
1,"(20.0, 30.0]",488969,883,24.314836,0.001806
2,"(30.0, 40.0]",131549,1142,33.832585,0.008681
3,"(40.0, 50.0]",42829,1777,44.196204,0.041491
4,"(50.0, 60.0]",12134,1914,54.020461,0.157739
5,"(60.0, 70.0]",3105,951,63.923331,0.306280
6,"(70.0, 80.0]",566,188,72.987143,0.332155
7,"(80.0, 90.0]",16,9,82.509340,0.562500


In [25]:
# Check whether higher behavioural risk actually corresponds to higher fraud.

behaviour_summary = (
    risk_df
    .groupby(
        pd.qcut(
            risk_df["behavioural_risk_score"],
            q=10,
            duplicates="drop"
        ),
        observed=True
    )
    .agg(
        transactions=("trans_num", "count"),
        fraud_cases=("is_fraud", "sum"),
        avg_behavioural_risk=("behavioural_risk_score", "mean")
    )
    .reset_index()
)

behaviour_summary["fraud_rate"] = (
    behaviour_summary["fraud_cases"]
    / behaviour_summary["transactions"]
)

behaviour_summary

,behavioural_risk_score,transactions,fraud_cases,avg_behavioural_risk,fraud_rate
0,"(0.0377, 6.022]",129668,141,3.791360,0.001087
1,"(6.022, 9.461]",129667,162,7.766542,0.001249
2,"(9.461, 12.283]",129668,149,10.788717,0.001149
3,"(12.283, 15.636]",129667,176,13.899934,0.001357
4,"(15.636, 19.22]",129668,198,17.441296,0.001527
5,"(19.22, 21.946]",129667,211,20.556638,0.001627
6,"(21.946, 24.907]",129667,335,23.465611,0.002584
7,"(24.907, 28.718]",129668,452,26.727877,0.003486
8,"(28.718, 33.966]",129667,764,31.099910,0.005892
9,"(33.966, 89.688]",129668,4918,41.187156,0.037928


In [26]:
# Check anomaly risk versus fraud.

anomaly_summary = (
    risk_df
    .groupby(
        pd.qcut(
            risk_df["anomaly_risk"],
            q=10,
            duplicates="drop"
        ),
        observed=True
    )
    .agg(
        transactions=("trans_num", "count"),
        fraud_cases=("is_fraud", "sum"),
        avg_anomaly_risk=("anomaly_risk", "mean")
    )
    .reset_index()
)

anomaly_summary["fraud_rate"] = (
    anomaly_summary["fraud_cases"]
    / anomaly_summary["transactions"]
)

anomaly_summary

,anomaly_risk,transactions,fraud_cases,avg_anomaly_risk,fraud_rate
0,"(-0.001, 11.904]",129668,75,9.186730,0.000578
1,"(11.904, 14.981]",129667,119,13.524220,0.000918
2,"(14.981, 17.442]",129668,147,16.235434,0.001134
3,"(17.442, 19.736]",129667,155,18.591584,0.001195
4,"(19.736, 22.073]",129668,162,20.896623,0.001249
5,"(22.073, 24.616]",129667,186,23.317841,0.001434
6,"(24.616, 27.626]",129667,207,26.063410,0.001596
7,"(27.626, 31.682]",129668,277,29.529239,0.002136
8,"(31.682, 38.735]",129667,692,34.773506,0.005337
9,"(38.735, 100.0]",129668,5486,48.258265,0.042308


In [27]:
risk_df[["anomaly_score", "anomaly_risk", "is_fraud"]].corr(numeric_only=True)

,anomaly_score,anomaly_risk,is_fraud
anomaly_score,1.000000,1.000000,0.184428
anomaly_risk,1.000000,1.000000,0.184428
is_fraud,0.184428,0.184428,1.000000


In [28]:
risk_df.groupby(
    pd.qcut(risk_df["anomaly_risk"], 10, duplicates="drop"),
    observed=True
)["is_fraud"].mean()

anomaly_risk
(-0.001, 11.904]    0.000578
(11.904, 14.981]    0.000918
(14.981, 17.442]    0.001134
(17.442, 19.736]    0.001195
(19.736, 22.073]    0.001249
(22.073, 24.616]    0.001434
(24.616, 27.626]    0.001596
(27.626, 31.682]    0.002136
(31.682, 38.735]    0.005337
(38.735, 100.0]     0.042308
Name: is_fraud, dtype: float64

In [29]:
risk_df.groupby(
    pd.qcut(risk_df["behavioural_risk_score"], 10, duplicates="drop"),
    observed=True
)["is_fraud"].mean()

behavioural_risk_score
(0.0377, 6.022]     0.001087
(6.022, 9.461]      0.001249
(9.461, 12.283]     0.001149
(12.283, 15.636]    0.001357
(15.636, 19.22]     0.001527
(19.22, 21.946]     0.001627
(21.946, 24.907]    0.002584
(24.907, 28.718]    0.003486
(28.718, 33.966]    0.005892
(33.966, 89.688]    0.037928
Name: is_fraud, dtype: float64

In [30]:
# ---------------------------------------------------------
# Final Risk Engine Validation
# ---------------------------------------------------------

validation_summary = pd.DataFrame({
    "metric": [
        "Total transactions",
        "Total fraud cases",
        "Overall fraud rate",
        "Low-risk fraud rate",
        "Medium-risk fraud rate",
        "High-risk fraud rate",
        "Critical-risk fraud rate",
        "Investigation rate"
    ],
    "value": [
        len(risk_df),
        risk_df["is_fraud"].sum(),
        risk_df["is_fraud"].mean(),
        risk_df.loc[
            risk_df["risk_band"] == "Low", "is_fraud"
        ].mean(),
        risk_df.loc[
            risk_df["risk_band"] == "Medium", "is_fraud"
        ].mean(),
        risk_df.loc[
            risk_df["risk_band"] == "High", "is_fraud"
        ].mean(),
        risk_df.loc[
            risk_df["risk_band"] == "Critical", "is_fraud"
        ].mean(),
        risk_df["investigation_flag"].mean()
    ]
})

validation_summary

,metric,value
0,Total transactions,1.296675e+06
1,Total fraud cases,7.506000e+03
2,Overall fraud rate,5.788652e-03
3,Low-risk fraud rate,1.378250e-03
4,Medium-risk fraud rate,2.591254e-02
5,High-risk fraud rate,3.102697e-01
6,Critical-risk fraud rate,5.625000e-01
7,Investigation rate,6.031581e-03


In [31]:
# Final risk-engine output
final_risk_df = risk_df[
    [
        "trans_num",
        "amount_risk",
        "velocity_risk",
        "novelty_risk",
        "behavioural_risk_score",
        "anomaly_score",
        "anomaly_risk",
        "final_risk_score",
        "risk_band",
        "investigation_flag",
        "is_fraud"
    ]
].copy()

print("Final risk-engine dataset:")
print(final_risk_df.head())

print("\nFinal columns:")
print(final_risk_df.columns.tolist())

print("\nRisk-band distribution:")
print(final_risk_df["risk_band"].value_counts())

Final risk-engine dataset:
                          trans_num  amount_risk  velocity_risk  novelty_risk  \
0  98e3dcf98101146a577f85a34e58feec     0.000000       0.000000         100.0   
1  498120fc45d277f7c88e3dba79c33865    97.611425       7.142857         100.0   
2  95f514bb993151347c7acdf8505c3d62    36.546841      14.285714          50.0   
3  4f0c1a14e0aa7eb56a490780ef9268c5     9.832239      21.428571         100.0   
4  3b2ebd3af508afba959640893e1e82bc     8.229834      21.428571         100.0   

   behavioural_risk_score  anomaly_score  anomaly_risk  final_risk_score  \
0               33.333333       0.027990     47.048750         38.819500   
1               68.251427       0.073429     61.673698         65.620336   
2               33.610852      -0.003784     36.822051         34.895332   
3               43.753604       0.011825     41.846073         42.990591   
4               43.219468       0.016785     43.442481         43.308674   

  risk_band  investigation_fl

In [32]:
# Save final risk-engine output for downstream use

OUTPUT_PATH = DATA_PATH / "sparkov_risk_engine.parquet"

final_risk_df.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(f"\nRisk-engine output saved to: {OUTPUT_PATH}")


Risk-engine output saved to: ..\data\processed\sparkov_risk_engine.parquet
